# BERT KMeans 多类别归属补充实验

本 notebook 用于补充方向 4 中“一条评论可能归属多个类别”的问题。

已有的 `BERT embedding + KMeans` 结果是单标签聚类：每条评论只属于一个 cluster。这里不重新训练 BERT，也不重新执行 KMeans，而是基于已有 embedding 和已有 cluster 结果，计算每条评论与各簇语义中心的余弦相似度。

输出方式：

```text
主类别：沿用原始 KMeans 分配的 cluster
候选次类别：从其他 cluster 中选取语义相似度最高的两个类别
```

这样可以形成“主类别 + 次类别”的多类别归属结果，用于缓解酒店评论中一条评论同时涉及服务、餐饮、房间、交通等多个方面的问题。

## 1. 导入依赖与定位项目目录

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root():
    """Find the project root that contains bert_kmeans_result."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for root in candidates:
        if (root / "bert_kmeans_result").exists():
            return root
    raise FileNotFoundError("Cannot find bert_kmeans_result from current working directory.")


PROJECT_ROOT = find_project_root()
RESULT_DIR = PROJECT_ROOT / "bert_kmeans_result"

CLUSTER_PATH = RESULT_DIR / "guangzhou_garden_hotel_bert_kmeans_clusters.csv"
EMBEDDING_PATH = RESULT_DIR / "comment_embeddings_bge_small_zh_v1_5.npy"

MULTILABEL_PATH = RESULT_DIR / "guangzhou_garden_hotel_bert_kmeans_multilabel.csv"
SUMMARY_PATH = RESULT_DIR / "guangzhou_garden_hotel_bert_kmeans_multilabel_summary.csv"

print("Project root:", PROJECT_ROOT)
print("Result dir:", RESULT_DIR)

Project root: /root/task4
Result dir: /root/task4/bert_kmeans_result


## 2. 读取已有 BERT KMeans 结果

In [2]:
df = pd.read_csv(CLUSTER_PATH)
embeddings = np.load(EMBEDDING_PATH)

if len(df) != len(embeddings):
    raise ValueError(f"Row count mismatch: df={len(df)}, embeddings={len(embeddings)}")

if "bert_kmeans_cluster" not in df.columns:
    raise ValueError("Missing column: bert_kmeans_cluster")

print("Comments:", len(df))
print("Embedding shape:", embeddings.shape)
display(df[["_id", "comment", "score", "quality_score", "bert_kmeans_cluster"]].head())

Comments: 2171
Embedding shape: (2171, 512)


,_id,comment,score,quality_score,bert_kmeans_cluster
0,68027895e3c98b0941765706,房间非常好 装修很厚重奢华 一开始看评论 看酒店自己po的照片 感觉跟快捷酒店一样 有些害怕...,5.0,9,3
1,68027895e3c98b0941765707,花园酒店广州的老牌五星，一直期望的入住。房间我入住时要求前台帮我升级了新装修的房间，粤韵。其...,4.2,8,3
2,68027895e3c98b0941765708,一进酒店就感觉来到了桃花源，迎宾人员都很热情，彬彬有礼，房间也非常不错，床品都很舒服，洗嗽用...,5.0,8,3
3,68027895e3c98b0941765709,总体来说很好。早餐有两个地方可以用，最上层的旋转餐厅用早餐感觉还是非常不错的。另外酒店的后花...,5.0,9,1
4,68027895e3c98b094176570a,多年没来广州，临时起意过来住花园酒店，环境舒适，没有想象中的老式，全员服务在线，礼貌周到，给...,5.0,9,3


## 3. 定义现有 5 个簇的类别名称

类别名称来自 `bert_kmeans_result_analysis.md` 中对 BERT KMeans 聚类结果的人工命名。

In [3]:
CATEGORY_NAMES = {
    0: "花园特色、文化底蕴与空间体验",
    1: "餐饮早餐、景观与基础住宿体验",
    2: "前台服务、舒适住宿与升级好评",
    3: "老牌五星综合长评体验",
    4: "问题反馈与负面复杂体验",
}

clusters = sorted(df["bert_kmeans_cluster"].unique().tolist())
print("Clusters:", clusters)

missing_names = [cluster for cluster in clusters if cluster not in CATEGORY_NAMES]
if missing_names:
    raise ValueError(f"Missing category names for clusters: {missing_names}")

pd.DataFrame([
    {"cluster": cluster, "category": CATEGORY_NAMES[cluster], "count": int((df["bert_kmeans_cluster"] == cluster).sum())}
    for cluster in clusters
])

Clusters: [0, 1, 2, 3, 4]


,cluster,category,count
0,0,花园特色、文化底蕴与空间体验,350
1,1,餐饮早餐、景观与基础住宿体验,526
2,2,前台服务、舒适住宿与升级好评,399
3,3,老牌五星综合长评体验,588
4,4,问题反馈与负面复杂体验,308


## 4. 计算每条评论到各类别中心的语义相似度

这里使用已有 cluster 中所有评论 embedding 的均值作为该类别的语义中心。然后计算每条评论 embedding 与每个类别中心的余弦相似度。

In [4]:
def l2_normalize(matrix, axis=1, eps=1e-12):
    norm = np.linalg.norm(matrix, axis=axis, keepdims=True)
    return matrix / np.maximum(norm, eps)


labels = df["bert_kmeans_cluster"].to_numpy()
embeddings_norm = l2_normalize(embeddings.astype(np.float32), axis=1)

centroids = []
for cluster in clusters:
    centroid = embeddings_norm[labels == cluster].mean(axis=0)
    centroids.append(centroid)

centroids = l2_normalize(np.vstack(centroids), axis=1)
similarities = embeddings_norm @ centroids.T

cluster_to_col = {cluster: idx for idx, cluster in enumerate(clusters)}

print("Similarity matrix:", similarities.shape)
print("Similarity range:", float(similarities.min()), float(similarities.max()))

Similarity matrix: (2171, 5)
Similarity range: 0.3781101107597351 0.921939492225647


## 5. 生成“主类别 + 候选次类别”结果

主类别沿用原始 KMeans 分配结果；候选次类别从其他类别中按相似度从高到低选取。

In [5]:
TOP_SECONDARY_N = 2

rows = []
for i, row in df.iterrows():
    primary_cluster = int(row["bert_kmeans_cluster"])
    primary_col = cluster_to_col[primary_cluster]
    primary_similarity = float(similarities[i, primary_col])

    candidate_cols = [col for col, cluster in enumerate(clusters) if cluster != primary_cluster]
    candidate_cols = sorted(candidate_cols, key=lambda col: similarities[i, col], reverse=True)
    secondary_cols = candidate_cols[:TOP_SECONDARY_N]
    secondary_clusters = [int(clusters[col]) for col in secondary_cols]
    secondary_categories = [CATEGORY_NAMES[cluster] for cluster in secondary_clusters]
    secondary_similarities = [float(similarities[i, col]) for col in secondary_cols]

    top_clusters = [primary_cluster] + secondary_clusters
    top_categories = [CATEGORY_NAMES[primary_cluster]] + secondary_categories
    top_similarities = [primary_similarity] + secondary_similarities

    output_row = row.to_dict()
    output_row.update({
        "primary_cluster": primary_cluster,
        "primary_category": CATEGORY_NAMES[primary_cluster],
        "primary_similarity": round(primary_similarity, 6),
        "secondary_cluster_1": secondary_clusters[0],
        "secondary_category_1": secondary_categories[0],
        "secondary_similarity_1": round(secondary_similarities[0], 6),
        "secondary_cluster_2": secondary_clusters[1],
        "secondary_category_2": secondary_categories[1],
        "secondary_similarity_2": round(secondary_similarities[1], 6),
        "primary_secondary_gap_1": round(primary_similarity - secondary_similarities[0], 6),
        "top3_clusters": "、".join(map(str, top_clusters)),
        "top3_categories": "、".join(top_categories),
        "top3_similarities": "、".join(f"{value:.6f}" for value in top_similarities),
    })
    rows.append(output_row)

df_multilabel = pd.DataFrame(rows)

display(df_multilabel[[
    "_id",
    "comment",
    "primary_category",
    "primary_similarity",
    "secondary_category_1",
    "secondary_similarity_1",
    "secondary_category_2",
    "secondary_similarity_2",
    "primary_secondary_gap_1",
]].head(10))

,_id,comment,primary_category,primary_similarity,secondary_category_1,secondary_similarity_1,secondary_category_2,secondary_similarity_2,primary_secondary_gap_1
0,68027895e3c98b0941765706,房间非常好 装修很厚重奢华 一开始看评论 看酒店自己po的照片 感觉跟快捷酒店一样 有些害怕...,老牌五星综合长评体验,0.836942,问题反馈与负面复杂体验,0.796852,餐饮早餐、景观与基础住宿体验,0.792362,0.040090
1,68027895e3c98b0941765707,花园酒店广州的老牌五星，一直期望的入住。房间我入住时要求前台帮我升级了新装修的房间，粤韵。其...,老牌五星综合长评体验,0.906153,问题反馈与负面复杂体验,0.840313,花园特色、文化底蕴与空间体验,0.828451,0.065840
2,68027895e3c98b0941765708,一进酒店就感觉来到了桃花源，迎宾人员都很热情，彬彬有礼，房间也非常不错，床品都很舒服，洗嗽用...,老牌五星综合长评体验,0.857380,餐饮早餐、景观与基础住宿体验,0.826424,花园特色、文化底蕴与空间体验,0.799306,0.030956
3,68027895e3c98b0941765709,总体来说很好。早餐有两个地方可以用，最上层的旋转餐厅用早餐感觉还是非常不错的。另外酒店的后花...,餐饮早餐、景观与基础住宿体验,0.841578,花园特色、文化底蕴与空间体验,0.805771,前台服务、舒适住宿与升级好评,0.777420,0.035807
4,68027895e3c98b094176570a,多年没来广州，临时起意过来住花园酒店，环境舒适，没有想象中的老式，全员服务在线，礼貌周到，给...,老牌五星综合长评体验,0.888021,问题反馈与负面复杂体验,0.821310,花园特色、文化底蕴与空间体验,0.802259,0.066711
5,68027895e3c98b094176570b,酒店环境很好，房间干净整洁，视野不错。酒店的后花园挺漂亮的，可以逛逛。附近买吃的也方便。\n...,花园特色、文化底蕴与空间体验,0.787713,餐饮早餐、景观与基础住宿体验,0.769794,问题反馈与负面复杂体验,0.764853,0.017919
6,68027895e3c98b094176570c,有集团金卡可以免费升级房型，酒店装修过非常的漂亮，后花园很大，景色很好。早餐非常好吃，电梯很...,餐饮早餐、景观与基础住宿体验,0.844693,花园特色、文化底蕴与空间体验,0.841508,前台服务、舒适住宿与升级好评,0.812834,0.003186
7,68027895e3c98b094176570d,酒店房间上放着餐厅有75折扣优惠，并且没有写有截至时间，但是去餐厅用完餐就被告知已经没有优惠...,问题反馈与负面复杂体验,0.759979,餐饮早餐、景观与基础住宿体验,0.711114,前台服务、舒适住宿与升级好评,0.686072,0.048865
8,68027895e3c98b094176570e,酒店服务相当不错，很有礼貌，入住和退房是挺快的。入住时还送了两朵康乃馨。不值钱，但是心意是有...,问题反馈与负面复杂体验,0.854996,餐饮早餐、景观与基础住宿体验,0.836775,花园特色、文化底蕴与空间体验,0.808431,0.018221
9,68027895e3c98b094176570f,房间硬件不错，服务、餐饮不及格，沟通好的生日主题布置，居然忘记了！提醒前台以后特意离开房间给...,餐饮早餐、景观与基础住宿体验,0.783425,问题反馈与负面复杂体验,0.768530,老牌五星综合长评体验,0.755254,0.014895


## 6. 查看可能的多主题评论

`primary_secondary_gap_1` 越小，说明主类别和第一候选次类别的相似度越接近，这类评论更可能是多主题评论。

In [6]:
mixed_candidates = df_multilabel.sort_values("primary_secondary_gap_1").head(20)

display(mixed_candidates[[
    "_id",
    "score",
    "comment",
    "primary_category",
    "secondary_category_1",
    "primary_similarity",
    "secondary_similarity_1",
    "primary_secondary_gap_1",
]])

,_id,score,comment,primary_category,secondary_category_1,primary_similarity,secondary_similarity_1,primary_secondary_gap_1
1914,68028542e3c98b09417660df,4.6,来广州出差选择最多的酒店之一，设施设备维护妥当，床品的质量也不错，不足的地方在于健身房的设备...,问题反馈与负面复杂体验,老牌五星综合长评体验,0.773197,0.776491,-0.003294
1322,68027f57e3c98b0941765ccf,4.7,服务有求必应，交通方便，动静脉分流合理，一楼早晚自助餐品种还行。建议大楼外玻璃清洗干净，看外...,问题反馈与负面复杂体验,餐饮早餐、景观与基础住宿体验,0.747149,0.749375,-0.002227
1579,680280c7e3c98b0941765e09,4.0,卫生减一星是杯子没消毒倒水有奇怪甜味\n来广州核心区，周边小吃餐厅餐馆众多\n楼下还有广式点...,问题反馈与负面复杂体验,餐饮早餐、景观与基础住宿体验,0.723425,0.724771,-0.001347
1359,68027f8be3c98b0941765cfe,5.0,^o^他们的床很舒服，枕头还有n款给你选，周边交通也方便，楼下就是地铁,问题反馈与负面复杂体验,前台服务、舒适住宿与升级好评,0.713154,0.714440,-0.001285
668,68027bf9e3c98b09417659ee,5.0,环境人文都很不错，就是住的楼层高的话窗户是封闭的，其他都很好,问题反馈与负面复杂体验,餐饮早餐、景观与基础住宿体验,0.728731,0.729717,-0.000986
379,68027a75e3c98b09417658a0,5.0,本次入住很满意，不愧是老牌五星级酒店，各方面都做的不错,餐饮早餐、景观与基础住宿体验,前台服务、舒适住宿与升级好评,0.847152,0.847761,-0.000609
564,68027b6de3c98b0941765976,4.7,美丽的花园酒店，广州的国际化大酒店，很美，服务也很好，特意表扬一下酒店的康乐部，游泳池的服务...,花园特色、文化底蕴与空间体验,问题反馈与负面复杂体验,0.849166,0.849699,-0.000532
1479,68028043e3c98b0941765d9b,5.0,酒店服务很好，下车就有人帮拿行李，大堂大气，办理入住井然有序，我非周末节假日去的正好人不算多...,花园特色、文化底蕴与空间体验,餐饮早餐、景观与基础住宿体验,0.864674,0.864884,-0.000210
590,68027b8fe3c98b0941765993,5.0,老牌酒店的品质保证，第一次入住花园酒店，感觉很贴心，免费升级了套房，送了入住果盘、小孩洗漱套...,老牌五星综合长评体验,花园特色、文化底蕴与空间体验,0.892448,0.892569,-0.000121
944,68027d58e3c98b0941765b1c,5.0,前台S小姐姐（忘了英文名了，只记得首字母是S）给免费升级到了套房，虽位于闹市，但房间很安静，...,问题反馈与负面复杂体验,前台服务、舒适住宿与升级好评,0.761412,0.761442,-0.000031


## 7. 汇总次类别分布

In [7]:
summary_rows = []
for primary_cluster in clusters:
    subset = df_multilabel[df_multilabel["primary_cluster"] == primary_cluster]
    for secondary_col in ["secondary_cluster_1", "secondary_cluster_2"]:
        counts = subset[secondary_col].value_counts().sort_index()
        for secondary_cluster, count in counts.items():
            summary_rows.append({
                "primary_cluster": int(primary_cluster),
                "primary_category": CATEGORY_NAMES[int(primary_cluster)],
                "secondary_rank": secondary_col.replace("secondary_cluster_", ""),
                "secondary_cluster": int(secondary_cluster),
                "secondary_category": CATEGORY_NAMES[int(secondary_cluster)],
                "count": int(count),
                "ratio_in_primary": round(int(count) / len(subset), 4),
            })

df_summary = pd.DataFrame(summary_rows)
display(df_summary.sort_values(["primary_cluster", "secondary_rank", "count"], ascending=[True, True, False]))

,primary_cluster,primary_category,secondary_rank,secondary_cluster,secondary_category,count,ratio_in_primary
0,0,花园特色、文化底蕴与空间体验,1,1,餐饮早餐、景观与基础住宿体验,182,0.5200
1,0,花园特色、文化底蕴与空间体验,1,2,前台服务、舒适住宿与升级好评,74,0.2114
2,0,花园特色、文化底蕴与空间体验,1,3,老牌五星综合长评体验,61,0.1743
3,0,花园特色、文化底蕴与空间体验,1,4,问题反馈与负面复杂体验,33,0.0943
4,0,花园特色、文化底蕴与空间体验,2,1,餐饮早餐、景观与基础住宿体验,121,0.3457
5,0,花园特色、文化底蕴与空间体验,2,2,前台服务、舒适住宿与升级好评,93,0.2657
6,0,花园特色、文化底蕴与空间体验,2,3,老牌五星综合长评体验,80,0.2286
7,0,花园特色、文化底蕴与空间体验,2,4,问题反馈与负面复杂体验,56,0.1600
9,1,餐饮早餐、景观与基础住宿体验,1,2,前台服务、舒适住宿与升级好评,202,0.3840
8,1,餐饮早餐、景观与基础住宿体验,1,0,花园特色、文化底蕴与空间体验,174,0.3308


## 8. 保存结果

In [8]:
df_multilabel.to_csv(MULTILABEL_PATH, index=False)
df_summary.to_csv(SUMMARY_PATH, index=False)

print("Saved multilabel result:", MULTILABEL_PATH)
print("Saved summary:", SUMMARY_PATH)

Saved multilabel result: /root/task4/bert_kmeans_result/guangzhou_garden_hotel_bert_kmeans_multilabel.csv
Saved summary: /root/task4/bert_kmeans_result/guangzhou_garden_hotel_bert_kmeans_multilabel_summary.csv
